In [ ]:
from pathlib import Path

dataset_folder = Path(
    "/kaggle/input/competitions/"
    "enveda-CASMI26-molecule-id-mass-spectra"
)

train_path = dataset_folder / "train.parquet"
test_path = dataset_folder / "test.parquet"
submission_path = dataset_folder / "sample_submission.csv"

print("Train file exists:", train_path.exists())
print("Test file exists:", test_path.exists())
print("Sample submission exists:", submission_path.exists())

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

test_file = pq.ParquetFile(test_path)
sample_submission = pd.read_csv(submission_path)

print("Test spectra:", test_file.metadata.num_rows)
print("Test columns:", test_file.schema_arrow.names)
print("Submission columns:", sample_submission.columns.tolist())
print("Sample submission rows:", len(sample_submission))

In [ ]:
# CASMI 2026 - Prepare test spectra for exact matching

import hashlib
import numpy as np
from collections import defaultdict

test_columns = [
    "molecule_id",
    "spectrum_id",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

test_spectra = test_file.read(
    columns=test_columns
).to_pandas()


def spectrum_fingerprint(row):
    mz = np.asarray(row.ms2_mzs, dtype=np.float64)

    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=np.float64
    )

    fingerprint = hashlib.blake2b(
        mz.tobytes() + intensity.tobytes(),
        digest_size=16
    ).digest()

    return (row.adduct, len(mz), fingerprint)


# Connect each fingerprint to its test molecule and spectrum
test_lookup = defaultdict(list)

for row in test_spectra.itertuples(index=False):
    fingerprint = spectrum_fingerprint(row)

    test_lookup[fingerprint].append(
        (row.molecule_id, row.spectrum_id)
    )


print("CASMI 2026 - Test Spectra Prepared")
print("----------------------------------")
print("Test spectra:", len(test_spectra))
print(
    "Unique test molecules:",
    test_spectra["molecule_id"].nunique()
)
print("Unique spectrum fingerprints:", len(test_lookup))

print(
    "Submission molecule IDs match test data:",
    set(sample_submission["molecule_id"])
    == set(test_spectra["molecule_id"])
)

print("\nTest spectra prepared successfully!")

In [ ]:
# CASMI 2026 - Find exact matches in the training dataset

import pyarrow.parquet as pq
from collections import defaultdict

train_file = pq.ParquetFile(train_path)

train_columns = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Skip spectra with adduct/peak-count combinations
# that do not occur in the test data.
test_combinations = {
    (row.adduct, len(row.ms2_mzs))
    for row in test_spectra.itertuples(index=False)
}

# molecule_id -> (inchikey14, SMILES) -> matched spectrum IDs
exact_hits = defaultdict(lambda: defaultdict(set))

print("CASMI 2026 - Exact Match Search")
print("-------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=10000,
        columns=train_columns
    ):
        batch_df = batch.to_pandas()

        for row in batch_df.itertuples(index=False):

            combination = (
                row.adduct,
                len(row.ms2_mzs)
            )

            if combination not in test_combinations:
                continue

            fingerprint = spectrum_fingerprint(row)

            for molecule_id, spectrum_id in test_lookup.get(
                fingerprint, []
            ):
                exact_hits[molecule_id][
                    (row.inchikey14, row.normalized_smiles)
                ].add(spectrum_id)

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )


# Summarize without assuming that all test molecules matched
test_molecule_count = test_spectra["molecule_id"].nunique()

matched_molecules = len(exact_hits)

unique_matches = sum(
    len(structures) == 1
    for structures in exact_hits.values()
)

ambiguous_matches = sum(
    len(structures) > 1
    for structures in exact_hits.values()
)

print("\nExact matching results:")
print("Test molecules:", test_molecule_count)
print("Molecules with exact matches:", matched_molecules)
print("Molecules with one matched structure:", unique_matches)
print("Molecules with multiple matched structures:", ambiguous_matches)
print(
    "Molecules without an exact match:",
    test_molecule_count - matched_molecules
)

In [ ]:
# CASMI 2026 - Prepare exact-match predictions

exact_predictions = {}
unresolved_molecules = []

for molecule_id in sample_submission["molecule_id"]:

    matched_structures = exact_hits.get(molecule_id, {})

    # Use an exact match only when it identifies one structure
    if len(matched_structures) == 1:

        inchikey14, smiles = next(
            iter(matched_structures.keys())
        )

        if (
            isinstance(smiles, str)
            and smiles.strip()
            and ";" not in smiles
        ):
            exact_predictions[molecule_id] = smiles
        else:
            unresolved_molecules.append(molecule_id)

    else:
        unresolved_molecules.append(molecule_id)


print("CASMI 2026 - Exact Predictions")
print("------------------------------")
print("Submission molecules:", len(sample_submission))
print("Exact predictions:", len(exact_predictions))
print("Unresolved molecules:", len(unresolved_molecules))

print("\nFirst 3 exact predictions:")
for molecule_id, smiles in list(exact_predictions.items())[:3]:
    print(molecule_id, "->", smiles)

print("\nExact prediction mapping prepared!")

In [ ]:
# CASMI 2026 - Build a fallback candidate library

import pandas as pd

library_columns = [
    "inchikey14",
    "normalized_smiles",
    "molecular_formula"
]

seen_keys = set()
library_parts = []

for batch in train_file.iter_batches(
    batch_size=20000,
    columns=library_columns
):
    batch_df = batch.to_pandas()

    # Remove incomplete rows and repeated molecular structures
    batch_df = batch_df.dropna(
        subset=library_columns
    ).drop_duplicates(subset="inchikey14")

    new_rows = batch_df[
        ~batch_df["inchikey14"].isin(seen_keys)
    ]

    if not new_rows.empty:
        library_parts.append(new_rows)
        seen_keys.update(new_rows["inchikey14"])

full_candidate_library = pd.concat(
    library_parts,
    ignore_index=True
)

print("CASMI 2026 - Fallback Candidate Library")
print("---------------------------------------")
print("Unique molecular structures:", len(full_candidate_library))
print(
    "Missing SMILES:",
    full_candidate_library["normalized_smiles"].isna().sum()
)
print(
    "Missing molecular formulas:",
    full_candidate_library["molecular_formula"].isna().sum()
)

print("\nCandidate library prepared!")

In [ ]:
# CASMI 2026 - Calculate candidate reference masses

import re
import numpy as np

# Monoisotopic atomic masses in Da
atomic_masses = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620,
    "S": 31.972071174,
    "P": 30.973761998,
    "F": 18.998403163,
    "Cl": 34.968852682,
    "Br": 78.918337600,
    "I": 126.904468000,
    "B": 11.009305360
}


def formula_to_reference_mass(formula):
    if not isinstance(formula, str):
        return np.nan

    # Remove trailing charge notation, where present
    elemental_formula = re.sub(r"\+\d*$", "", formula)

    parts = re.findall(
        r"([A-Z][a-z]?)(\d*)",
        elemental_formula
    )

    # Reject formulas we cannot parse completely
    reconstructed = "".join(
        element + count
        for element, count in parts
    )

    if reconstructed != elemental_formula or not parts:
        return np.nan

    if any(
        element not in atomic_masses
        for element, _ in parts
    ):
        return np.nan

    return sum(
        atomic_masses[element] * int(count or 1)
        for element, count in parts
    )


full_candidate_library["reference_mass"] = (
    full_candidate_library["molecular_formula"]
    .apply(formula_to_reference_mass)
)

print("CASMI 2026 - Reference Mass Calculation")
print("---------------------------------------")
print("Total candidates:", len(full_candidate_library))
print(
    "Candidates with reference mass:",
    int(full_candidate_library["reference_mass"].notna().sum())
)
print(
    "Candidates with missing reference mass:",
    int(full_candidate_library["reference_mass"].isna().sum())
)

print("\nReference mass calculation completed!")

In [ ]:
# CASMI 2026 - Calculate neutral masses for test molecules

import pandas as pd

adduct_shifts = {
    "[M+H]+": 1.007276466621,
    "[M-H]-": -1.007276466621,
    "[M+CH2O2-H]-": 44.99820284,
    "[M+Na]+": 22.989218,
    "[M+NH4]+": 18.033823,
    "[M+K]+": 38.963158,
    "[M+Cl]-": 34.969401
}

# Read the test metadata available during this notebook run
mass_data = test_file.read(
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "precursor_mz"
    ]
).to_pandas()

mass_data["adduct_shift"] = (
    mass_data["adduct"].map(adduct_shifts)
)

mass_data["neutral_mass"] = (
    pd.to_numeric(mass_data["precursor_mz"], errors="coerce")
    - mass_data["adduct_shift"]
)

# Calculate one median neutral mass per molecule
test_molecules = (
    mass_data.groupby("molecule_id", as_index=False)
    .agg(
        median_neutral_mass=("neutral_mass", "median"),
        num_spectra=("spectrum_id", "count")
    )
)

print("CASMI 2026 - Test Neutral Masses")
print("--------------------------------")
print("Test spectra:", len(mass_data))
print("Test molecules:", len(test_molecules))

print(
    "Spectra with unknown adducts:",
    int(mass_data["adduct_shift"].isna().sum())
)

print(
    "Molecules with missing neutral mass:",
    int(test_molecules["median_neutral_mass"].isna().sum())
)

print("\nFirst 5 molecules:")
print(test_molecules.head().to_string(index=False))

In [ ]:
# CASMI 2026 - Generate submission.csv

import numpy as np
import pandas as pd
from pathlib import Path

# Prepare the mass-based candidate library
mass_library = (
    full_candidate_library
    .dropna(subset=["reference_mass", "normalized_smiles"])
    .sort_values("reference_mass")
    .reset_index(drop=True)
)

sorted_masses = mass_library["reference_mass"].to_numpy(
    dtype=float
)
sorted_smiles = mass_library["normalized_smiles"].to_numpy()

mass_lookup = test_molecules.set_index(
    "molecule_id"
)["median_neutral_mass"]

submission_rows = []

for molecule_id in sample_submission["molecule_id"]:

    selected_smiles = []
    seen_smiles = set()

    # Place an unambiguous exact match first, if available
    exact_smiles = exact_predictions.get(molecule_id)

    if exact_smiles is not None:
        selected_smiles.append(exact_smiles)
        seen_smiles.add(exact_smiles)

    # Fill remaining positions using nearby reference masses
    query_mass = float(mass_lookup.loc[molecule_id])

    if not np.isfinite(query_mass):
        raise ValueError(
            f"No usable neutral mass for {molecule_id}"
        )

    right = int(np.searchsorted(sorted_masses, query_mass))
    left = right - 1

    while len(selected_smiles) < 25:

        if left < 0:
            candidate_index = right
            right += 1

        elif right >= len(sorted_masses):
            candidate_index = left
            left -= 1

        else:
            left_error = abs(
                sorted_masses[left] - query_mass
            )
            right_error = abs(
                sorted_masses[right] - query_mass
            )

            if left_error <= right_error:
                candidate_index = left
                left -= 1
            else:
                candidate_index = right
                right += 1

        candidate_smiles = sorted_smiles[candidate_index]

        if candidate_smiles not in seen_smiles:
            selected_smiles.append(candidate_smiles)
            seen_smiles.add(candidate_smiles)

    submission_rows.append({
        "molecule_id": molecule_id,
        "smiles": ";".join(selected_smiles)
    })

submission = pd.DataFrame(submission_rows)

# Check the required format
assert submission.columns.tolist() == ["molecule_id", "smiles"]
assert len(submission) == len(sample_submission)

assert (
    submission["molecule_id"].tolist()
    == sample_submission["molecule_id"].tolist()
)

assert all(
    len(smiles.split(";")) == 25
    and len(set(smiles.split(";"))) == 25
    for smiles in submission["smiles"]
)

# Save using the filename required by Kaggle
output_path = Path("/kaggle/working/submission.csv")
submission.to_csv(output_path, index=False)

print("CASMI 2026 - Submission Created")
print("--------------------------------")
print("File exists:", output_path.exists())
print("File path:", output_path)
print("Submission rows:", len(submission))
print("SMILES per molecule: 25")
print(
    "Molecules with exact prediction:",
    sum(
        molecule_id in exact_predictions
        for molecule_id in submission["molecule_id"]
    )
)
print(
    "Molecules using mass-only fallback:",
    sum(
        molecule_id not in exact_predictions
        for molecule_id in submission["molecule_id"]
    )
)
print("\nSubmission format validation passed!")

In [10]:
# CASMI 2026 - Evaluate the current mass-only fallback

import numpy as np

# Use the variables already created in our submission notebook:
# sorted_masses, sorted_smiles, mass_lookup, exact_predictions

total = 0
top1_matches = 0
top25_matches = 0
reciprocal_rank_sum = 0.0

for molecule_id in sample_submission["molecule_id"]:

    known_smiles = exact_predictions.get(molecule_id)

    if known_smiles is None:
        continue

    query_mass = float(mass_lookup.loc[molecule_id])

    if not np.isfinite(query_mass):
        continue

    # Recreate the mass-only ranking without using the exact match
    selected_smiles = []
    seen_smiles = set()

    right = int(np.searchsorted(sorted_masses, query_mass))
    left = right - 1

    while len(selected_smiles) < 25:

        if left < 0:
            candidate_index = right
            right += 1

        elif right >= len(sorted_masses):
            candidate_index = left
            left -= 1

        else:
            left_error = abs(sorted_masses[left] - query_mass)
            right_error = abs(sorted_masses[right] - query_mass)

            if left_error <= right_error:
                candidate_index = left
                left -= 1
            else:
                candidate_index = right
                right += 1

        candidate_smiles = sorted_smiles[candidate_index]

        if candidate_smiles not in seen_smiles:
            selected_smiles.append(candidate_smiles)
            seen_smiles.add(candidate_smiles)

    total += 1

    if known_smiles in selected_smiles:
        rank = selected_smiles.index(known_smiles) + 1
        top25_matches += 1
        reciprocal_rank_sum += 1.0 / rank

        if rank == 1:
            top1_matches += 1

print("CASMI 2026 - Mass-Only Fallback Diagnostic")
print("------------------------------------------")
print("Molecules evaluated:", total)
print("Exact-match SMILES at rank 1:", top1_matches)
print("Exact-match SMILES within top 25:", top25_matches)

if total:
    print("Top-1 rate:", round(top1_matches / total, 4))
    print("Top-25 rate:", round(top25_matches / total, 4))
    print("Mean Reciprocal Rank @ 25:", round(
        reciprocal_rank_sum / total, 4
    ))

CASMI 2026 - Mass-Only Fallback Diagnostic
------------------------------------------
Molecules evaluated: 400
Exact-match SMILES at rank 1: 23
Exact-match SMILES within top 25: 221
Top-1 rate: 0.0575
Top-25 rate: 0.5525
Mean Reciprocal Rank @ 25: 0.1386


In [11]:
# CASMI 2026 - Diagnose mass-only fallback misses

import numpy as np

# Find reference masses associated with each SMILES
masses_by_smiles = {}

for row in full_candidate_library[
    ["normalized_smiles", "reference_mass"]
].itertuples(index=False):

    if np.isfinite(row.reference_mass):
        masses_by_smiles.setdefault(
            row.normalized_smiles, []
        ).append(float(row.reference_mass))


diagnostic = {
    "correct_smiles_not_in_library": 0,
    "correct_mass_within_20_ppm": 0,
    "correct_mass_above_20_ppm": 0,
    "missed_top25_but_mass_within_20_ppm": 0
}

examples = []

for molecule_id in sample_submission["molecule_id"]:

    true_smiles = exact_predictions[molecule_id]
    query_mass = float(mass_lookup.loc[molecule_id])

    reference_masses = masses_by_smiles.get(true_smiles)

    if not reference_masses:
        diagnostic["correct_smiles_not_in_library"] += 1
        continue

    best_mass_error_ppm = min(
        abs(reference_mass - query_mass)
        / query_mass * 1_000_000
        for reference_mass in reference_masses
    )

    if best_mass_error_ppm <= 20:
        diagnostic["correct_mass_within_20_ppm"] += 1
    else:
        diagnostic["correct_mass_above_20_ppm"] += 1

    # Is the correct SMILES in our previous mass-only top 25?
    predicted_smiles = submission.loc[
        submission["molecule_id"] == molecule_id,
        "smiles"
    ].iloc[0].split(";")

    # Remove the exact-match prediction from consideration.
    # The remaining entries are not necessarily identical to
    # the mass-only top 25, so use the mass-only search below.
    left = int(np.searchsorted(sorted_masses, query_mass)) - 1
    right = left + 1
    mass_only_top25 = []
    seen = set()

    while len(mass_only_top25) < 25:
        if left < 0:
            index = right
            right += 1
        elif right >= len(sorted_masses):
            index = left
            left -= 1
        elif abs(sorted_masses[left] - query_mass) <= abs(
            sorted_masses[right] - query_mass
        ):
            index = left
            left -= 1
        else:
            index = right
            right += 1

        smiles = sorted_smiles[index]

        if smiles not in seen:
            mass_only_top25.append(smiles)
            seen.add(smiles)

    if (
        true_smiles not in mass_only_top25
        and best_mass_error_ppm <= 20
    ):
        diagnostic["missed_top25_but_mass_within_20_ppm"] += 1

        if len(examples) < 5:
            examples.append(
                (molecule_id, round(best_mass_error_ppm, 4))
            )


print("CASMI 2026 - Mass-Only Diagnostic")
print("--------------------------------")
for label, value in diagnostic.items():
    print(f"{label}: {value}")

print("\nExamples missed from top 25 despite mass match:")
for molecule_id, error_ppm in examples:
    print(molecule_id, "| mass error:", error_ppm, "ppm")

CASMI 2026 - Mass-Only Diagnostic
--------------------------------
correct_smiles_not_in_library: 0
correct_mass_within_20_ppm: 400
correct_mass_above_20_ppm: 0
missed_top25_but_mass_within_20_ppm: 179

Examples missed from top 25 despite mass match:
m_006153 | mass error: 1.2033 ppm
m_00b5aa | mass error: 2.2235 ppm
m_0259d4 | mass error: 2.0597 ppm
m_02d188 | mass error: 0.6344 ppm
m_050bf1 | mass error: 2.161 ppm


In [12]:
# CASMI 2026 - Prepare references without exact test-spectrum copies

import numpy as np
import pandas as pd

molecule_id = "m_006153"
ppm_tolerance = 20

# Test spectra and observed adducts for this molecule
queries = test_spectra[
    test_spectra["molecule_id"] == molecule_id
].copy()

test_adducts = set(queries["adduct"])

# Identify the structure found through exact matching
true_key, true_smiles = next(
    iter(exact_hits[molecule_id].keys())
)

# Find all mass-matched candidate structures
query_mass = float(
    test_molecules.loc[
        test_molecules["molecule_id"] == molecule_id,
        "median_neutral_mass"
    ].iloc[0]
)

ppm_errors = (
    abs(full_candidate_library["reference_mass"] - query_mass)
    / query_mass
) * 1_000_000

candidate_keys = set(
    full_candidate_library.loc[
        ppm_errors <= ppm_tolerance,
        "inchikey14"
    ]
)

# Fingerprints of this molecule's test spectra
query_fingerprints = {
    spectrum_fingerprint(row)
    for row in queries.itertuples(index=False)
}

reference_parts = []
excluded_exact_copies = 0

# Read training data in batches
for batch in train_file.iter_batches(
    batch_size=20000,
    columns=[
        "inchikey14",
        "adduct",
        "ms2_mzs",
        "ms2_normalized_intensities"
    ]
):
    batch_df = batch.to_pandas()

    matches = batch_df[
        batch_df["inchikey14"].isin(candidate_keys)
        & batch_df["adduct"].isin(test_adducts)
    ].copy()

    if matches.empty:
        continue

    # Exclude reference spectra identical to any query spectrum
    is_exact_copy = matches.apply(
        lambda row: spectrum_fingerprint(row)
        in query_fingerprints,
        axis=1
    )

    excluded_exact_copies += int(is_exact_copy.sum())

    remaining = matches.loc[~is_exact_copy]

    if not remaining.empty:
        reference_parts.append(remaining)

experiment_references = (
    pd.concat(reference_parts, ignore_index=True)
    if reference_parts
    else pd.DataFrame()
)

print("CASMI 2026 - Spectral Ranking Experiment")
print("----------------------------------------")
print("Test molecule:", molecule_id)
print("Test spectra:", len(queries))
print("Mass-matched candidates:", len(candidate_keys))
print("Exact test-spectrum copies excluded:", excluded_exact_copies)
print("Remaining reference spectra:", len(experiment_references))

if not experiment_references.empty:
    print(
        "Candidates with remaining references:",
        experiment_references["inchikey14"].nunique()
    )
    print(
        "Remaining references for the known structure:",
        int(
            (
                experiment_references["inchikey14"]
                == true_key
            ).sum()
        )
    )

CASMI 2026 - Spectral Ranking Experiment
----------------------------------------
Test molecule: m_006153
Test spectra: 6
Mass-matched candidates: 257
Exact test-spectrum copies excluded: 6
Remaining reference spectra: 1376
Candidates with remaining references: 250
Remaining references for the known structure: 1


In [13]:
# CASMI 2026 - Rank candidates without exact spectrum copies

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Convert a spectrum into 1 Da intensity bins
def extract_features(row):
    features = np.zeros(1000, dtype=np.float32)

    mzs = np.asarray(row["ms2_mzs"], dtype=float)
    intensities = np.asarray(
        row["ms2_normalized_intensities"],
        dtype=float
    )

    for mz, intensity in zip(mzs, intensities):
        if np.isfinite(mz) and np.isfinite(intensity):
            index = int(mz)
            if 0 <= index < 1000:
                features[index] = max(
                    features[index], intensity
                )

    return features


# Prepare all mass-matched candidates
ranking = full_candidate_library[
    full_candidate_library["inchikey14"].isin(candidate_keys)
].copy()

ranking["mass_error_ppm"] = (
    abs(ranking["reference_mass"] - query_mass)
    / query_mass * 1_000_000
)

# Give every candidate a score for every query spectrum.
# A missing matching-adduct reference contributes zero.
query_count = len(queries)

score_sums = {
    key: 0.0 for key in candidate_keys
}

matched_query_counts = {
    key: 0 for key in candidate_keys
}

# Compare only spectra with the same adduct
for adduct, query_group in queries.groupby("adduct"):

    ref_group = experiment_references[
        experiment_references["adduct"] == adduct
    ]

    if ref_group.empty:
        continue

    X_query = np.stack(
        query_group.apply(
            extract_features, axis=1
        ).to_numpy()
    )

    X_ref = np.stack(
        ref_group.apply(
            extract_features, axis=1
        ).to_numpy()
    )

    similarity = cosine_similarity(X_query, X_ref)
    ref_keys = ref_group["inchikey14"].to_numpy()

    for key in np.unique(ref_keys):

        positions = np.flatnonzero(ref_keys == key)

        # Best reference match for each query spectrum
        best_scores = similarity[:, positions].max(axis=1)

        score_sums[key] += float(best_scores.sum())
        matched_query_counts[key] += len(best_scores)


ranking["spectral_score"] = ranking["inchikey14"].map(
    lambda key: (
        score_sums[key] / query_count
        if matched_query_counts[key] > 0
        else np.nan
    )
)

ranking["matched_query_spectra"] = (
    ranking["inchikey14"].map(matched_query_counts)
)

ranking = ranking.sort_values(
    ["spectral_score", "mass_error_ppm"],
    ascending=[False, True],
    na_position="last"
).reset_index(drop=True)

# Evaluate the known structure AFTER ranking
known_positions = np.flatnonzero(
    ranking["inchikey14"].to_numpy() == true_key
)

known_rank = (
    int(known_positions[0]) + 1
    if len(known_positions) == 1
    else None
)

print("CASMI 2026 - Spectral Ranking Experiment")
print("----------------------------------------")
print("Test molecule:", molecule_id)
print("Total candidates:", len(ranking))
print(
    "Candidates with spectral scores:",
    int(ranking["spectral_score"].notna().sum())
)
print("Known structure rank:", known_rank)
print(
    "Known structure in top 25:",
    known_rank is not None and known_rank <= 25
)

print("\nTop 5 candidates:")
print(
    ranking[
        [
            "inchikey14",
            "spectral_score",
            "matched_query_spectra",
            "mass_error_ppm"
        ]
    ].head(5).to_string(index=False)
)

if known_rank is not None:
    print("\nKnown structure:")
    print(
        ranking.loc[
            known_rank - 1,
            [
                "inchikey14",
                "spectral_score",
                "matched_query_spectra",
                "mass_error_ppm"
            ]
        ].to_string()
    )

CASMI 2026 - Spectral Ranking Experiment
----------------------------------------
Test molecule: m_006153
Total candidates: 257
Candidates with spectral scores: 250
Known structure rank: 15
Known structure in top 25: True

Top 5 candidates:
    inchikey14  spectral_score  matched_query_spectra  mass_error_ppm
OWLWDUZRCJYSHF        0.443144                      3        1.203321
JFPHIOHAVGPTKT        0.371177                      6       15.639837
SSOWPHRKXKLBRF        0.357845                      6        1.203321
VYMUBUVCGNOAMN        0.328112                      6       11.719535
MDQHIPKMJWCDLW        0.327811                      6        8.369637

Known structure:
inchikey14               BKIHGKNEJKYYAC
spectral_score                 0.246311
matched_query_spectra                 3
mass_error_ppm                 1.203321


In [14]:
# CASMI 2026 - Inspect the known candidate's spectral scores

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

known_refs = experiment_references[
    experiment_references["inchikey14"] == true_key
]

print("CASMI 2026 - Known Structure Spectral Scores")
print("---------------------------------------------")
print("Test molecule:", molecule_id)
print("Known structure:", true_key)
print("Reference spectra:", len(known_refs))

for _, query_row in queries.iterrows():

    same_adduct_refs = known_refs[
        known_refs["adduct"] == query_row["adduct"]
    ]

    if same_adduct_refs.empty:
        print(
            query_row["spectrum_id"],
            "| Adduct:", query_row["adduct"],
            "| No matching reference"
        )
        continue

    query_vector = extract_features(query_row).reshape(1, -1)

    reference_vectors = np.stack(
        same_adduct_refs.apply(
            extract_features,
            axis=1
        ).to_numpy()
    )

    best_score = cosine_similarity(
        query_vector,
        reference_vectors
    ).max()

    print(
        query_row["spectrum_id"],
        "| Adduct:", query_row["adduct"],
        "| Best similarity:", round(float(best_score), 6)
    )

CASMI 2026 - Known Structure Spectral Scores
---------------------------------------------
Test molecule: m_006153
Known structure: BKIHGKNEJKYYAC
Reference spectra: 1
s_3c74e9fe | Adduct: [M+H]+ | Best similarity: 0.148354
s_54245904 | Adduct: [M-H]- | No matching reference
s_59a80abf | Adduct: [M-H]- | No matching reference
s_7151fa86 | Adduct: [M+H]+ | Best similarity: 0.592624
s_86472581 | Adduct: [M-H]- | No matching reference
s_cb15f0ac | Adduct: [M+H]+ | Best similarity: 0.736886


In [15]:
# CASMI 2026 - Compare three spectral scoring methods

import numpy as np
import pandas as pd

comparison = ranking[
    [
        "inchikey14",
        "mass_error_ppm",
        "spectral_score",
        "matched_query_spectra"
    ]
].copy()

# Method 1: Original score
# Missing query-adduct references contribute zero.
comparison["score_original"] = comparison["spectral_score"]

# Method 2: Average over query spectra that have references.
# This does not penalize missing reference adducts.
comparison["score_available"] = comparison.apply(
    lambda row: (
        score_sums[row["inchikey14"]]
        / matched_query_counts[row["inchikey14"]]
        if matched_query_counts[row["inchikey14"]] > 0
        else np.nan
    ),
    axis=1
)

# Method 3: Average over available spectra,
# with a moderate penalty for incomplete reference coverage.
coverage = comparison["matched_query_spectra"] / len(queries)

comparison["score_coverage_adjusted"] = (
    comparison["score_available"]
    * (0.5 + 0.5 * coverage)
)

print("CASMI 2026 - Scoring Method Comparison")
print("--------------------------------------")
print("Test molecule:", molecule_id)
print("Known structure:", true_key)

for method in [
    "score_original",
    "score_available",
    "score_coverage_adjusted"
]:
    ordered = comparison.sort_values(
        [method, "mass_error_ppm"],
        ascending=[False, True],
        na_position="last"
    ).reset_index(drop=True)

    positions = np.flatnonzero(
        ordered["inchikey14"].to_numpy() == true_key
    )

    known_rank = (
        int(positions[0]) + 1
        if len(positions) == 1
        else None
    )

    print(f"\nMethod: {method}")
    print("Known structure rank:", known_rank)

    if known_rank is not None:
        print(
            "Known structure score:",
            round(
                float(ordered.loc[known_rank - 1, method]),
                6
            )
        )

    print(
        "Top candidate:",
        ordered.loc[0, "inchikey14"]
    )

CASMI 2026 - Scoring Method Comparison
--------------------------------------
Test molecule: m_006153
Known structure: BKIHGKNEJKYYAC

Method: score_original
Known structure rank: 15
Known structure score: 0.246311
Top candidate: OWLWDUZRCJYSHF

Method: score_available
Known structure rank: 4
Known structure score: 0.492621
Top candidate: OWLWDUZRCJYSHF

Method: score_coverage_adjusted
Known structure rank: 5
Known structure score: 0.369466
Top candidate: OWLWDUZRCJYSHF


In [16]:
# CASMI 2026 - Check memory before batch spectral evaluation

import os
import psutil

memory = psutil.virtual_memory()
process = psutil.Process(os.getpid())

print("CASMI 2026 - Memory Check")
print("-------------------------")
print(
    "Total RAM:",
    round(memory.total / (1024 ** 3), 2),
    "GB"
)
print(
    "Available RAM:",
    round(memory.available / (1024 ** 3), 2),
    "GB"
)
print(
    "Notebook RAM usage:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)
print(
    "Training row groups:",
    train_file.num_row_groups
)

CASMI 2026 - Memory Check
-------------------------
Total RAM: 31.35 GB
Available RAM: 23.64 GB
Notebook RAM usage: 6.49 GB
Training row groups: 21


In [17]:
# CASMI 2026 - Prepare a reproducible evaluation subset

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

all_molecule_ids = sample_submission["molecule_id"].to_numpy()

evaluation_ids = rng.choice(
    all_molecule_ids,
    size=min(20, len(all_molecule_ids)),
    replace=False
).tolist()

# Use the same 20 ppm mass filter for every molecule
library_with_mass = full_candidate_library.dropna(
    subset=["reference_mass"]
)

reference_masses = library_with_mass["reference_mass"].to_numpy(
    dtype=float
)

evaluation_candidates = {}
all_candidate_keys = set()
all_evaluation_adducts = set()

for molecule_id in evaluation_ids:

    query_mass = float(mass_lookup.loc[molecule_id])

    ppm_errors = (
        np.abs(reference_masses - query_mass)
        / query_mass
    ) * 1_000_000

    candidates = library_with_mass.loc[
        ppm_errors <= 20
    ]

    candidate_keys = set(candidates["inchikey14"])

    evaluation_candidates[molecule_id] = candidate_keys
    all_candidate_keys.update(candidate_keys)

    molecule_adducts = set(
        test_spectra.loc[
            test_spectra["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    all_evaluation_adducts.update(molecule_adducts)


print("CASMI 2026 - Batch Evaluation Preparation")
print("-----------------------------------------")
print("Evaluation molecules:", len(evaluation_ids))

print(
    "Evaluation test spectra:",
    int(
        test_spectra["molecule_id"]
        .isin(evaluation_ids)
        .sum()
    )
)

print(
    "Unique candidate structures to search:",
    len(all_candidate_keys)
)

print(
    "Minimum candidates per molecule:",
    min(len(keys) for keys in evaluation_candidates.values())
)

print(
    "Maximum candidates per molecule:",
    max(len(keys) for keys in evaluation_candidates.values())
)

print(
    "Adducts in evaluation subset:",
    sorted(all_evaluation_adducts)
)

print("\nEvaluation molecules:")
print(evaluation_ids)

CASMI 2026 - Batch Evaluation Preparation
-----------------------------------------
Evaluation molecules: 20
Evaluation test spectra: 55
Unique candidate structures to search: 2306
Minimum candidates per molecule: 9
Maximum candidates per molecule: 347
Adducts in evaluation subset: ['[M+H]+', '[M+Na]+', '[M-H]-']

Evaluation molecules:
['m_c886af', 'm_bf6cfe', 'm_30024e', 'm_f866a0', 'm_e138be', 'm_c9a4bc', 'm_1de28b', 'm_153108', 'm_acdfec', 'm_8d19fa', 'm_7108be', 'm_145679', 'm_7b993b', 'm_dc7be7', 'm_15dab4', 'm_c3e52e', 'm_d07b86', 'm_741f2c', 'm_b8f328', 'm_8d75c7']


In [18]:
# CASMI 2026 - Small, memory-safe reference search test

import os
import psutil

process = psutil.Process(os.getpid())

memory_before = process.memory_info().rss / (1024 ** 3)

# Read only two metadata columns from the first 2,000 rows
small_batch = next(
    train_file.iter_batches(
        batch_size=2000,
        columns=["inchikey14", "adduct"]
    )
).to_pandas()

# Find candidate structures from our 20-molecule evaluation set
matches = small_batch[
    small_batch["inchikey14"].isin(all_candidate_keys)
    & small_batch["adduct"].isin(all_evaluation_adducts)
]

memory_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Safe Reference Search Test")
print("---------------------------------------")
print("Rows inspected:", len(small_batch))
print("Matching rows:", len(matches))
print("Unique matching structures:", matches["inchikey14"].nunique())
print("Notebook RAM before:", round(memory_before, 3), "GB")
print("Notebook RAM after:", round(memory_after, 3), "GB")
print("RAM change:", round(memory_after - memory_before, 3), "GB")

CASMI 2026 - Safe Reference Search Test
---------------------------------------
Rows inspected: 2000
Matching rows: 2
Unique matching structures: 2
Notebook RAM before: 6.494 GB
Notebook RAM after: 6.495 GB
RAM change: 0.0 GB


In [19]:
# CASMI 2026 - Bounded reference collection test

import os
import psutil
import pandas as pd

process = psutil.Process(os.getpid())

MAX_ROWS = 50000
MAX_REFERENCES = 500
BATCH_SIZE = 5000

memory_before = process.memory_info().rss / (1024 ** 3)

pilot_parts = []
rows_inspected = 0
references_collected = 0

columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Read a maximum of 10 batches from row group 0
for batch in train_file.iter_batches(
    row_groups=[0],
    batch_size=BATCH_SIZE,
    columns=columns
):
    batch_df = batch.to_pandas()

    rows_inspected += len(batch_df)

    matches = batch_df[
        batch_df["inchikey14"].isin(all_candidate_keys)
        & batch_df["adduct"].isin(all_evaluation_adducts)
    ]

    # Do not collect more than 500 reference spectra
    remaining_capacity = (
        MAX_REFERENCES - references_collected
    )

    if remaining_capacity > 0 and not matches.empty:
        selected = matches.head(remaining_capacity).copy()
        pilot_parts.append(selected)

        references_collected += len(selected)

    if (
        rows_inspected >= MAX_ROWS
        or references_collected >= MAX_REFERENCES
    ):
        break

pilot_references = (
    pd.concat(pilot_parts, ignore_index=True)
    if pilot_parts
    else pd.DataFrame(columns=columns)
)

memory_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Bounded Reference Collection")
print("-----------------------------------------")
print("Rows inspected:", rows_inspected)
print("Reference spectra collected:", len(pilot_references))
print(
    "Unique candidate structures:",
    pilot_references["inchikey14"].nunique()
)

print(
    "RAM before:",
    round(memory_before, 3),
    "GB"
)

print(
    "RAM after:",
    round(memory_after, 3),
    "GB"
)

print(
    "RAM change:",
    round(memory_after - memory_before, 3),
    "GB"
)

print(
    "Available RAM:",
    round(
        psutil.virtual_memory().available / (1024 ** 3),
        3
    ),
    "GB"
)

# Free the temporary references after measuring memory
del pilot_references
del pilot_parts
del batch_df

print("\nBounded collection test completed!")

CASMI 2026 - Bounded Reference Collection
-----------------------------------------
Rows inspected: 50000
Reference spectra collected: 379
Unique candidate structures: 92
RAM before: 6.495 GB
RAM after: 6.727 GB
RAM change: 0.232 GB
Available RAM: 23.409 GB

Bounded collection test completed!


In [20]:
# CASMI 2026 - Bounded reference collection test

import os
import psutil
import pandas as pd

process = psutil.Process(os.getpid())

MAX_ROWS = 50000
MAX_REFERENCES = 500
BATCH_SIZE = 5000

memory_before = process.memory_info().rss / (1024 ** 3)

pilot_parts = []
rows_inspected = 0
references_collected = 0

columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Read a maximum of 10 batches from row group 0
for batch in train_file.iter_batches(
    row_groups=[0],
    batch_size=BATCH_SIZE,
    columns=columns
):
    batch_df = batch.to_pandas()

    rows_inspected += len(batch_df)

    matches = batch_df[
        batch_df["inchikey14"].isin(all_candidate_keys)
        & batch_df["adduct"].isin(all_evaluation_adducts)
    ]

    # Do not collect more than 500 reference spectra
    remaining_capacity = (
        MAX_REFERENCES - references_collected
    )

    if remaining_capacity > 0 and not matches.empty:
        selected = matches.head(remaining_capacity).copy()
        pilot_parts.append(selected)

        references_collected += len(selected)

    if (
        rows_inspected >= MAX_ROWS
        or references_collected >= MAX_REFERENCES
    ):
        break

pilot_references = (
    pd.concat(pilot_parts, ignore_index=True)
    if pilot_parts
    else pd.DataFrame(columns=columns)
)

memory_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Bounded Reference Collection")
print("-----------------------------------------")
print("Rows inspected:", rows_inspected)
print("Reference spectra collected:", len(pilot_references))
print(
    "Unique candidate structures:",
    pilot_references["inchikey14"].nunique()
)

print(
    "RAM before:",
    round(memory_before, 3),
    "GB"
)

print(
    "RAM after:",
    round(memory_after, 3),
    "GB"
)

print(
    "RAM change:",
    round(memory_after - memory_before, 3),
    "GB"
)

print(
    "Available RAM:",
    round(
        psutil.virtual_memory().available / (1024 ** 3),
        3
    ),
    "GB"
)

# Free the temporary references after measuring memory
del pilot_references
del pilot_parts
del batch_df

print("\nBounded collection test completed!")

CASMI 2026 - Bounded Reference Collection
-----------------------------------------
Rows inspected: 50000
Reference spectra collected: 379
Unique candidate structures: 92
RAM before: 6.727 GB
RAM after: 6.696 GB
RAM change: -0.031 GB
Available RAM: 23.43 GB

Bounded collection test completed!


In [21]:
# CASMI 2026 - Memory-limited reference collection

import os
import gc
import psutil
import pandas as pd

process = psutil.Process(os.getpid())

BATCH_SIZE = 5000
MAX_REFERENCES = 40000
MAX_PROCESS_RAM_GB = 12.0
MIN_AVAILABLE_RAM_GB = 6.0

reference_columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Fingerprints of all test spectra in our evaluation subset
evaluation_queries = test_spectra[
    test_spectra["molecule_id"].isin(evaluation_ids)
]

evaluation_fingerprints = {
    spectrum_fingerprint(row)
    for row in evaluation_queries.itertuples(index=False)
}

reference_parts = []
references_collected = 0
exact_copies_excluded = 0
rows_inspected = 0

collection_complete = True
stop_reason = None

print("CASMI 2026 - Reference Collection")
print("---------------------------------")

for group_index in range(train_file.num_row_groups):

    memory = psutil.virtual_memory()
    ram_used = process.memory_info().rss / (1024 ** 3)
    ram_available = memory.available / (1024 ** 3)

    # Stop before reading another row group if memory is low
    if (
        ram_used >= MAX_PROCESS_RAM_GB
        or ram_available <= MIN_AVAILABLE_RAM_GB
    ):
        collection_complete = False
        stop_reason = "RAM safety limit reached"
        break

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=BATCH_SIZE,
        columns=reference_columns
    ):
        batch_df = batch.to_pandas()
        rows_inspected += len(batch_df)

        matches = batch_df[
            batch_df["inchikey14"].isin(all_candidate_keys)
            & batch_df["adduct"].isin(all_evaluation_adducts)
        ].copy()

        if not matches.empty:

            # Remove exact copies of evaluation test spectra
            is_copy = [
                spectrum_fingerprint(row)
                in evaluation_fingerprints
                for row in matches.itertuples(index=False)
            ]

            exact_copies_excluded += sum(is_copy)

            remaining = matches.loc[
                ~pd.Series(is_copy, index=matches.index)
            ]

            if not remaining.empty:
                reference_parts.append(remaining)
                references_collected += len(remaining)

        del batch_df, matches

        # Check memory after each batch
        ram_used = process.memory_info().rss / (1024 ** 3)
        ram_available = (
            psutil.virtual_memory().available / (1024 ** 3)
        )

        if references_collected >= MAX_REFERENCES:
            collection_complete = False
            stop_reason = "Reference count limit reached"
            break

        if (
            ram_used >= MAX_PROCESS_RAM_GB
            or ram_available <= MIN_AVAILABLE_RAM_GB
        ):
            collection_complete = False
            stop_reason = "RAM safety limit reached"
            break

    print(
        f"Row groups checked: {group_index + 1}/"
        f"{train_file.num_row_groups} | "
        f"References: {references_collected} | "
        f"RAM: {ram_used:.2f} GB"
    )

    gc.collect()

    if not collection_complete:
        break


evaluation_references = (
    pd.concat(reference_parts, ignore_index=True)
    if reference_parts
    else pd.DataFrame(columns=reference_columns)
)

del reference_parts
gc.collect()

print("\nCASMI 2026 - Collection Results")
print("--------------------------------")
print("Collection complete:", collection_complete)
print("Rows inspected:", rows_inspected)
print("References collected:", len(evaluation_references))
print(
    "Unique reference structures:",
    evaluation_references["inchikey14"].nunique()
)
print("Exact copies excluded:", exact_copies_excluded)

print(
    "Current notebook RAM:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)

if not collection_complete:
    print("Stopped early:", stop_reason)
else:
    print("All training row groups checked successfully!")

CASMI 2026 - Reference Collection
---------------------------------
Row groups checked: 1/21 | References: 997 | RAM: 6.99 GB
Row groups checked: 2/21 | References: 1920 | RAM: 7.44 GB
Row groups checked: 3/21 | References: 2791 | RAM: 7.94 GB
Row groups checked: 4/21 | References: 3767 | RAM: 8.41 GB
Row groups checked: 5/21 | References: 4713 | RAM: 8.88 GB
Row groups checked: 6/21 | References: 5468 | RAM: 9.41 GB
Row groups checked: 7/21 | References: 6380 | RAM: 9.88 GB
Row groups checked: 8/21 | References: 7281 | RAM: 10.39 GB
Row groups checked: 9/21 | References: 7987 | RAM: 10.85 GB
Row groups checked: 10/21 | References: 8282 | RAM: 11.40 GB
Row groups checked: 11/21 | References: 8531 | RAM: 11.50 GB
Row groups checked: 12/21 | References: 8742 | RAM: 11.91 GB
Row groups checked: 13/21 | References: 9234 | RAM: 11.95 GB
Row groups checked: 14/21 | References: 10056 | RAM: 11.79 GB
Row groups checked: 15/21 | References: 10847 | RAM: 11.87 GB
Row groups checked: 16/21 | Refe

In [22]:
# CASMI 2026 - Convert collected references to compact feature vectors

import gc
import os
import numpy as np
import psutil

process = psutil.Process(os.getpid())

ram_before = process.memory_info().rss / (1024 ** 3)

# Keep only the metadata needed for candidate ranking
reference_metadata = evaluation_references[
    ["inchikey14", "adduct"]
].copy().reset_index(drop=True)

reference_count = len(evaluation_references)

# Allocate a compact float32 matrix
reference_features = np.zeros(
    (reference_count, 1000),
    dtype=np.float32
)

# Convert one reference spectrum at a time
for i, row in enumerate(
    evaluation_references.itertuples(index=False)
):

    mz = np.asarray(row.ms2_mzs, dtype=float)

    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=float
    )

    valid = (
        np.isfinite(mz)
        & np.isfinite(intensity)
        & (mz >= 0)
        & (mz < 1000)
    )

    bins = mz[valid].astype(np.int32)

    # Retain the maximum intensity in each 1 Da bin
    np.maximum.at(
        reference_features[i],
        bins,
        intensity[valid].astype(np.float32)
    )

# Release the large DataFrame containing raw fragment arrays
del evaluation_references
gc.collect()

ram_after = process.memory_info().rss / (1024 ** 3)

print("CASMI 2026 - Compact Reference Features")
print("---------------------------------------")
print("Reference spectra converted:", reference_count)
print("Feature matrix shape:", reference_features.shape)

print(
    "Feature matrix size:",
    round(reference_features.nbytes / (1024 ** 2), 2),
    "MB"
)

print(
    "Metadata rows:",
    len(reference_metadata)
)

print("RAM before:", round(ram_before, 2), "GB")
print("RAM after:", round(ram_after, 2), "GB")

print("\nCompact reference features prepared!")

CASMI 2026 - Compact Reference Features
---------------------------------------
Reference spectra converted: 11798
Feature matrix shape: (11798, 1000)
Feature matrix size: 45.01 MB
Metadata rows: 11798
RAM before: 12.0 GB
RAM after: 12.05 GB

Compact reference features prepared!


In [1]:
# CASMI 2026 - Check notebook state before continuing

import os
import psutil

required_variables = [
    "train_file",
    "test_spectra",
    "evaluation_ids",
    "evaluation_candidates",
    "all_candidate_keys",
    "all_evaluation_adducts",
    "reference_features",
    "reference_metadata"
]

print("CASMI 2026 - Notebook State Check")
print("---------------------------------")

for variable_name in required_variables:
    exists = variable_name in globals()
    print(f"{variable_name}: {'Available' if exists else 'Missing'}")

process = psutil.Process(os.getpid())

print(
    "\nCurrent notebook RAM:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)

print(
    "Available Kaggle RAM:",
    round(psutil.virtual_memory().available / (1024 ** 3), 2),
    "GB"
)

CASMI 2026 - Notebook State Check
---------------------------------
train_file: Missing
test_spectra: Missing
evaluation_ids: Missing
evaluation_candidates: Missing
all_candidate_keys: Missing
all_evaluation_adducts: Missing
reference_features: Missing
reference_metadata: Missing

Current notebook RAM: 0.1 GB
Available Kaggle RAM: 30.12 GB


In [2]:
# CASMI 2026 - Restore essential notebook variables

from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

dataset_folder = Path(
    "/kaggle/input/competitions/"
    "enveda-CASMI26-molecule-id-mass-spectra"
)

train_path = dataset_folder / "train.parquet"
test_path = dataset_folder / "test.parquet"
submission_path = dataset_folder / "sample_submission.csv"

train_file = pq.ParquetFile(train_path)
test_file = pq.ParquetFile(test_path)

sample_submission = pd.read_csv(submission_path)

test_spectra = test_file.read(
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "ms2_mzs",
        "ms2_normalized_intensities"
    ]
).to_pandas()

print("CASMI 2026 - Essential Variables Restored")
print("-----------------------------------------")
print("Training row groups:", train_file.num_row_groups)
print("Test spectra:", len(test_spectra))
print("Test molecules:", test_spectra["molecule_id"].nunique())
print("Submission rows:", len(sample_submission))
print("\nRestoration successful!")

CASMI 2026 - Essential Variables Restored
-----------------------------------------
Training row groups: 21
Test spectra: 1213
Test molecules: 400
Submission rows: 400

Restoration successful!


In [3]:
# CASMI 2026 - Restore the same 20 evaluation molecules

import numpy as np

rng = np.random.default_rng(42)

evaluation_ids = rng.choice(
    sample_submission["molecule_id"].to_numpy(),
    size=min(20, len(sample_submission)),
    replace=False
).tolist()

# Verify that we restored the original evaluation subset
expected_ids = [
    "m_c886af", "m_bf6cfe", "m_30024e", "m_f866a0",
    "m_e138be", "m_c9a4bc", "m_1de28b", "m_153108",
    "m_acdfec", "m_8d19fa", "m_7108be", "m_145679",
    "m_7b993b", "m_dc7be7", "m_15dab4", "m_c3e52e",
    "m_d07b86", "m_741f2c", "m_b8f328", "m_8d75c7"
]

assert evaluation_ids == expected_ids, (
    "The evaluation subset differs from the previous experiment."
)

evaluation_queries = test_spectra[
    test_spectra["molecule_id"].isin(evaluation_ids)
].copy()

print("CASMI 2026 - Evaluation Subset Restored")
print("---------------------------------------")
print("Evaluation molecules:", len(evaluation_ids))
print("Evaluation test spectra:", len(evaluation_queries))
print("Original evaluation subset restored: True")

CASMI 2026 - Evaluation Subset Restored
---------------------------------------
Evaluation molecules: 20
Evaluation test spectra: 55
Original evaluation subset restored: True


In [4]:
# CASMI 2026 - Restore the candidate library

import pandas as pd

library_columns = [
    "inchikey14",
    "normalized_smiles",
    "molecular_formula"
]

seen_keys = set()
library_parts = []

print("CASMI 2026 - Restoring Candidate Library")
print("----------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=20000,
        columns=library_columns
    ):
        batch_df = batch.to_pandas()

        batch_df = batch_df.dropna(
            subset=library_columns
        ).drop_duplicates(subset="inchikey14")

        new_rows = batch_df[
            ~batch_df["inchikey14"].isin(seen_keys)
        ]

        if not new_rows.empty:
            library_parts.append(new_rows)
            seen_keys.update(new_rows["inchikey14"])

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )

full_candidate_library = pd.concat(
    library_parts,
    ignore_index=True
)

del library_parts

print("\nCASMI 2026 - Candidate Library Restored")
print("---------------------------------------")
print("Unique structures:", len(full_candidate_library))
print(
    "Duplicate structures:",
    int(full_candidate_library["inchikey14"].duplicated().sum())
)

print("\nRestoration completed!")

CASMI 2026 - Restoring Candidate Library
----------------------------------------
Row groups checked: 5/21
Row groups checked: 10/21
Row groups checked: 15/21
Row groups checked: 20/21
Row groups checked: 21/21

CASMI 2026 - Candidate Library Restored
---------------------------------------
Unique structures: 275810
Duplicate structures: 0

Restoration completed!


In [5]:
# CASMI 2026 - Restore candidate reference masses

import re
import numpy as np

atomic_masses = {
    "C": 12.000000000,
    "H": 1.007825032,
    "N": 14.003074004,
    "O": 15.994914620,
    "S": 31.972071174,
    "P": 30.973761998,
    "F": 18.998403163,
    "Cl": 34.968852682,
    "Br": 78.918337600,
    "I": 126.904468000,
    "B": 11.009305360
}

def formula_to_reference_mass(formula):
    if not isinstance(formula, str):
        return np.nan

    elemental_formula = re.sub(r"\+\d*$", "", formula)

    parts = re.findall(
        r"([A-Z][a-z]?)(\d*)",
        elemental_formula
    )

    reconstructed = "".join(
        element + count
        for element, count in parts
    )

    if reconstructed != elemental_formula or not parts:
        return np.nan

    if any(
        element not in atomic_masses
        for element, _ in parts
    ):
        return np.nan

    return sum(
        atomic_masses[element] * int(count or 1)
        for element, count in parts
    )

full_candidate_library["reference_mass"] = (
    full_candidate_library["molecular_formula"]
    .apply(formula_to_reference_mass)
)

print("CASMI 2026 - Reference Masses Restored")
print("--------------------------------------")
print("Total candidates:", len(full_candidate_library))
print(
    "Candidates with reference mass:",
    int(full_candidate_library["reference_mass"].notna().sum())
)
print(
    "Candidates with missing reference mass:",
    int(full_candidate_library["reference_mass"].isna().sum())
)

CASMI 2026 - Reference Masses Restored
--------------------------------------
Total candidates: 275810
Candidates with reference mass: 275763
Candidates with missing reference mass: 47


In [6]:
# CASMI 2026 - Restore test neutral masses

import pandas as pd

adduct_shifts = {
    "[M+H]+": 1.007276466621,
    "[M-H]-": -1.007276466621,
    "[M+CH2O2-H]-": 44.99820284,
    "[M+Na]+": 22.989218,
    "[M+NH4]+": 18.033823,
    "[M+K]+": 38.963158,
    "[M+Cl]-": 34.969401
}

mass_data = test_file.read(
    columns=[
        "molecule_id",
        "spectrum_id",
        "adduct",
        "precursor_mz"
    ]
).to_pandas()

mass_data["adduct_shift"] = (
    mass_data["adduct"].map(adduct_shifts)
)

mass_data["neutral_mass"] = (
    pd.to_numeric(
        mass_data["precursor_mz"],
        errors="coerce"
    ) - mass_data["adduct_shift"]
)

test_molecules = (
    mass_data.groupby("molecule_id", as_index=False)
    .agg(
        median_neutral_mass=("neutral_mass", "median"),
        num_spectra=("spectrum_id", "count")
    )
)

mass_lookup = test_molecules.set_index(
    "molecule_id"
)["median_neutral_mass"]

print("CASMI 2026 - Neutral Masses Restored")
print("------------------------------------")
print("Test molecules:", len(test_molecules))
print(
    "Missing neutral masses:",
    int(test_molecules["median_neutral_mass"].isna().sum())
)
print(
    "Evaluation molecules with valid masses:",
    int(mass_lookup.loc[evaluation_ids].notna().sum())
)

CASMI 2026 - Neutral Masses Restored
------------------------------------
Test molecules: 400
Missing neutral masses: 0
Evaluation molecules with valid masses: 20


In [7]:
# CASMI 2026 - Restore candidates for the 20-molecule experiment

import numpy as np

PPM_TOLERANCE = 20

library_with_mass = full_candidate_library.dropna(
    subset=["reference_mass"]
)

reference_masses = library_with_mass[
    "reference_mass"
].to_numpy(dtype=float)

evaluation_candidates = {}
all_candidate_keys = set()
all_evaluation_adducts = set()

for molecule_id in evaluation_ids:

    query_mass = float(mass_lookup.loc[molecule_id])

    ppm_errors = (
        np.abs(reference_masses - query_mass)
        / query_mass
    ) * 1_000_000

    candidate_keys = set(
        library_with_mass.loc[
            ppm_errors <= PPM_TOLERANCE,
            "inchikey14"
        ]
    )

    evaluation_candidates[molecule_id] = candidate_keys
    all_candidate_keys.update(candidate_keys)

    molecule_adducts = set(
        test_spectra.loc[
            test_spectra["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    all_evaluation_adducts.update(molecule_adducts)

print("CASMI 2026 - Evaluation Candidates Restored")
print("-------------------------------------------")
print("Evaluation molecules:", len(evaluation_candidates))
print("Unique candidate structures:", len(all_candidate_keys))
print(
    "Minimum candidates per molecule:",
    min(len(keys) for keys in evaluation_candidates.values())
)
print(
    "Maximum candidates per molecule:",
    max(len(keys) for keys in evaluation_candidates.values())
)
print("Adducts:", sorted(all_evaluation_adducts))

assert len(evaluation_candidates) == 20
assert len(all_candidate_keys) == 2306

print("\nOriginal candidate set restored successfully!")

CASMI 2026 - Evaluation Candidates Restored
-------------------------------------------
Evaluation molecules: 20
Unique candidate structures: 2306
Minimum candidates per molecule: 9
Maximum candidates per molecule: 347
Adducts: ['[M+H]+', '[M+Na]+', '[M-H]-']

Original candidate set restored successfully!


In [8]:
# CASMI 2026 - Stream, compress and save evaluation references

import os
import gc
import hashlib
import numpy as np
import pandas as pd
import psutil

process = psutil.Process(os.getpid())

BATCH_SIZE = 5000
MAX_PROCESS_RAM_GB = 10.0
MIN_AVAILABLE_RAM_GB = 6.0
MAX_REFERENCES = 25000

reference_columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]


def spectrum_fingerprint(row):
    mz = np.asarray(row.ms2_mzs, dtype=np.float64)
    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=np.float64
    )

    digest = hashlib.blake2b(
        mz.tobytes() + intensity.tobytes(),
        digest_size=16
    ).digest()

    return (row.adduct, len(mz), digest)


def make_feature_vector(row):
    features = np.zeros(1000, dtype=np.float32)

    mz = np.asarray(row.ms2_mzs, dtype=float)
    intensity = np.asarray(
        row.ms2_normalized_intensities,
        dtype=float
    )

    valid = (
        np.isfinite(mz)
        & np.isfinite(intensity)
        & (mz >= 0)
        & (mz < 1000)
    )

    bins = mz[valid].astype(np.int32)

    np.maximum.at(
        features,
        bins,
        intensity[valid].astype(np.float32)
    )

    return features


# Exact copies must not enter our evaluation references
evaluation_fingerprints = {
    spectrum_fingerprint(row)
    for row in evaluation_queries.itertuples(index=False)
}

feature_parts = []
reference_keys = []
reference_adducts = []

excluded_copies = 0
rows_inspected = 0

collection_complete = True
stop_reason = None

print("CASMI 2026 - Compact Reference Collection")
print("-----------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=BATCH_SIZE,
        columns=reference_columns
    ):

        batch_df = batch.to_pandas()
        rows_inspected += len(batch_df)

        matches = batch_df[
            batch_df["inchikey14"].isin(all_candidate_keys)
            & batch_df["adduct"].isin(all_evaluation_adducts)
        ]

        for row in matches.itertuples(index=False):

            if spectrum_fingerprint(row) in evaluation_fingerprints:
                excluded_copies += 1
                continue

            if len(feature_parts) >= MAX_REFERENCES:
                collection_complete = False
                stop_reason = "Reference count limit reached"
                break

            feature_parts.append(make_feature_vector(row))
            reference_keys.append(row.inchikey14)
            reference_adducts.append(row.adduct)

        del matches, batch_df, batch

        # Check RAM after each batch
        used_ram = process.memory_info().rss / (1024 ** 3)
        available_ram = (
            psutil.virtual_memory().available / (1024 ** 3)
        )

        if (
            used_ram >= MAX_PROCESS_RAM_GB
            or available_ram <= MIN_AVAILABLE_RAM_GB
        ):
            collection_complete = False
            stop_reason = "RAM safety limit reached"
            break

        if not collection_complete:
            break

    print(
        f"Row groups checked: {group_index + 1}/"
        f"{train_file.num_row_groups} | "
        f"References: {len(feature_parts)} | "
        f"RAM: {used_ram:.2f} GB"
    )

    gc.collect()

    if not collection_complete:
        break


print("\nCASMI 2026 - Collection Results")
print("--------------------------------")
print("Collection complete:", collection_complete)
print("Rows inspected:", rows_inspected)
print("References collected:", len(feature_parts))
print("Exact copies excluded:", excluded_copies)
print("Current RAM:", round(
    process.memory_info().rss / (1024 ** 3), 2
), "GB")

if collection_complete:

    reference_features = np.stack(feature_parts)

    reference_metadata = pd.DataFrame({
        "inchikey14": reference_keys,
        "adduct": reference_adducts
    })

    output_file = (
        "/kaggle/working/"
        "casmi_evaluation_references.npz"
    )

    # Save features and metadata together for future sessions
    np.savez_compressed(
        output_file,
        features=reference_features,
        inchikey14=np.asarray(reference_keys),
        adduct=np.asarray(reference_adducts),
        evaluation_ids=np.asarray(evaluation_ids)
    )

    print(
        "Feature matrix shape:",
        reference_features.shape
    )
    print(
        "Feature matrix size:",
        round(reference_features.nbytes / (1024 ** 2), 2),
        "MB"
    )
    print("Saved file:", output_file)

else:
    print("Stopped early:", stop_reason)
    print("Incomplete results were NOT saved.")

del feature_parts
gc.collect()

CASMI 2026 - Compact Reference Collection
-----------------------------------------
Row groups checked: 1/21 | References: 997 | RAM: 0.68 GB
Row groups checked: 2/21 | References: 1920 | RAM: 0.69 GB
Row groups checked: 3/21 | References: 2791 | RAM: 0.69 GB
Row groups checked: 4/21 | References: 3767 | RAM: 0.69 GB
Row groups checked: 5/21 | References: 4713 | RAM: 0.72 GB
Row groups checked: 6/21 | References: 5468 | RAM: 0.73 GB
Row groups checked: 7/21 | References: 6380 | RAM: 0.73 GB
Row groups checked: 8/21 | References: 7281 | RAM: 0.74 GB
Row groups checked: 9/21 | References: 7987 | RAM: 0.75 GB
Row groups checked: 10/21 | References: 8282 | RAM: 0.79 GB
Row groups checked: 11/21 | References: 8531 | RAM: 0.79 GB
Row groups checked: 12/21 | References: 8742 | RAM: 0.84 GB
Row groups checked: 13/21 | References: 9234 | RAM: 0.85 GB
Row groups checked: 14/21 | References: 10056 | RAM: 0.85 GB
Row groups checked: 15/21 | References: 10847 | RAM: 0.85 GB
Row groups checked: 16/2

0

In [9]:
# CASMI 2026 - Restore evaluation labels for the same 20 molecules

from collections import defaultdict
import pandas as pd

# Index only the 55 evaluation test spectra
evaluation_lookup = defaultdict(set)

for row in evaluation_queries.itertuples(index=False):
    fingerprint = spectrum_fingerprint(row)
    evaluation_lookup[fingerprint].add(row.molecule_id)

# Store structures associated with exact training-spectrum matches
evaluation_labels = defaultdict(set)

columns = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

print("CASMI 2026 - Restoring Evaluation Labels")
print("---------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=5000,
        columns=columns
    ):
        batch_df = batch.to_pandas()

        for row in batch_df.itertuples(index=False):

            fingerprint = spectrum_fingerprint(row)

            for molecule_id in evaluation_lookup.get(
                fingerprint, ()
            ):
                evaluation_labels[molecule_id].add(
                    (row.inchikey14, row.normalized_smiles)
                )

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )


# Count molecules with unambiguous evaluation labels
unique_labels = {
    molecule_id: next(iter(structures))
    for molecule_id, structures in evaluation_labels.items()
    if len(structures) == 1
}

ambiguous_count = sum(
    len(structures) > 1
    for structures in evaluation_labels.values()
)

print("\nEvaluation molecules:", len(evaluation_ids))
print("Molecules with one matched structure:", len(unique_labels))
print("Molecules with multiple matched structures:", ambiguous_count)
print(
    "Molecules without a matched structure:",
    len(evaluation_ids) - len(evaluation_labels)
)

# Save labels only if all 20 molecules have one matched structure
if len(unique_labels) == len(evaluation_ids):

    evaluation_label_df = pd.DataFrame([
        {
            "molecule_id": molecule_id,
            "inchikey14": unique_labels[molecule_id][0],
            "normalized_smiles": unique_labels[molecule_id][1]
        }
        for molecule_id in evaluation_ids
    ])

    labels_path = "/kaggle/working/casmi_evaluation_labels.csv"

    evaluation_label_df.to_csv(
        labels_path,
        index=False
    )

    print("\nEvaluation labels saved:", labels_path)

else:
    print("\nSome labels are missing or ambiguous; check before evaluation.")

CASMI 2026 - Restoring Evaluation Labels
---------------------------------------
Row groups checked: 5/21
Row groups checked: 10/21
Row groups checked: 15/21
Row groups checked: 20/21
Row groups checked: 21/21

Evaluation molecules: 20
Molecules with one matched structure: 20
Molecules with multiple matched structures: 0
Molecules without a matched structure: 0

Evaluation labels saved: /kaggle/working/casmi_evaluation_labels.csv


In [10]:
# CASMI 2026 - Evaluate four ranking methods on 20 molecules

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Labels are used ONLY to evaluate completed rankings
labels = pd.read_csv(
    "/kaggle/working/casmi_evaluation_labels.csv"
).set_index("molecule_id")["inchikey14"]

ref_keys = reference_metadata["inchikey14"].to_numpy()
ref_adducts = reference_metadata["adduct"].to_numpy()

methods = [
    "mass_only",
    "original",
    "available",
    "coverage_adjusted"
]

results = []

print("CASMI 2026 - Evaluating 20 Molecules")
print("------------------------------------")

for molecule_id in evaluation_ids:

    query_group = evaluation_queries[
        evaluation_queries["molecule_id"] == molecule_id
    ]

    query_mass = float(mass_lookup.loc[molecule_id])
    candidate_keys = evaluation_candidates[molecule_id]
    known_key = labels.loc[molecule_id]

    # Candidate metadata and mass errors
    candidates = library_with_mass[
        library_with_mass["inchikey14"].isin(candidate_keys)
    ][["inchikey14", "reference_mass"]].copy()

    candidates["mass_error_ppm"] = (
        np.abs(candidates["reference_mass"] - query_mass)
        / query_mass * 1_000_000
    )

    score_sums = {key: 0.0 for key in candidate_keys}
    score_counts = {key: 0 for key in candidate_keys}

    # Select references for this molecule only
    candidate_mask = np.isin(
        ref_keys, list(candidate_keys)
    )

    for adduct, adduct_queries in query_group.groupby("adduct"):

        ref_positions = np.flatnonzero(
            candidate_mask & (ref_adducts == adduct)
        )

        if len(ref_positions) == 0:
            continue

        # Small matrices: only this molecule and this adduct
        X_query = np.stack([
            make_feature_vector(row)
            for row in adduct_queries.itertuples(index=False)
        ])

        X_ref = reference_features[ref_positions]

        similarities = cosine_similarity(X_query, X_ref)
        selected_keys = ref_keys[ref_positions]

        for key in np.unique(selected_keys):

            key_positions = np.flatnonzero(
                selected_keys == key
            )

            best_per_query = similarities[
                :, key_positions
            ].max(axis=1)

            score_sums[key] += float(best_per_query.sum())
            score_counts[key] += len(best_per_query)

    total_queries = len(query_group)

    candidates["matched_queries"] = candidates[
        "inchikey14"
    ].map(score_counts)

    candidates["original"] = candidates["inchikey14"].map(
        lambda key: (
            score_sums[key] / total_queries
            if score_counts[key] > 0
            else np.nan
        )
    )

    candidates["available"] = candidates["inchikey14"].map(
        lambda key: (
            score_sums[key] / score_counts[key]
            if score_counts[key] > 0
            else np.nan
        )
    )

    coverage = candidates["matched_queries"] / total_queries

    candidates["coverage_adjusted"] = (
        candidates["available"] * (0.5 + 0.5 * coverage)
    )

    # Evaluate each method AFTER ranking
    for method in methods:

        if method == "mass_only":
            ordered = candidates.sort_values(
                ["mass_error_ppm", "inchikey14"],
                ascending=[True, True]
            )
        else:
            ordered = candidates.sort_values(
                [method, "mass_error_ppm", "inchikey14"],
                ascending=[False, True, True],
                na_position="last"
            )

        ranked_keys = ordered["inchikey14"].to_numpy()

        positions = np.flatnonzero(ranked_keys == known_key)

        known_rank = (
            int(positions[0]) + 1
            if len(positions) == 1
            else None
        )

        results.append({
            "molecule_id": molecule_id,
            "method": method,
            "known_rank": known_rank,
            "reciprocal_rank_at_25": (
                1.0 / known_rank
                if known_rank is not None and known_rank <= 25
                else 0.0
            ),
            "top1": known_rank == 1,
            "top25": (
                known_rank is not None and known_rank <= 25
            )
        })

    print(f"Evaluated: {molecule_id}")

evaluation_results_df = pd.DataFrame(results)

summary = (
    evaluation_results_df.groupby("method", sort=False)
    .agg(
        molecules=("molecule_id", "count"),
        top1_matches=("top1", "sum"),
        top25_matches=("top25", "sum"),
        mean_reciprocal_rank_at_25=(
            "reciprocal_rank_at_25", "mean"
        )
    )
)

print("\nCASMI 2026 - Evaluation Results")
print("--------------------------------")
print(summary.to_string())

print("\nResults for m_006153:")
print(
    evaluation_results_df.loc[
        evaluation_results_df["molecule_id"] == "m_006153",
        ["method", "known_rank"]
    ].to_string(index=False)
)

CASMI 2026 - Evaluating 20 Molecules
------------------------------------
Evaluated: m_c886af
Evaluated: m_bf6cfe
Evaluated: m_30024e
Evaluated: m_f866a0
Evaluated: m_e138be
Evaluated: m_c9a4bc
Evaluated: m_1de28b
Evaluated: m_153108
Evaluated: m_acdfec
Evaluated: m_8d19fa
Evaluated: m_7108be
Evaluated: m_145679
Evaluated: m_7b993b
Evaluated: m_dc7be7
Evaluated: m_15dab4
Evaluated: m_c3e52e
Evaluated: m_d07b86
Evaluated: m_741f2c
Evaluated: m_b8f328
Evaluated: m_8d75c7

CASMI 2026 - Evaluation Results
--------------------------------
                   molecules  top1_matches  top25_matches  mean_reciprocal_rank_at_25
method                                                                               
mass_only                 20             1             13                    0.141274
original                  20             7             10                    0.416667
available                 20             5             10                    0.358333
coverage_adjusted         20  

In [11]:
# CASMI 2026 - Compare rankings molecule by molecule

import pandas as pd

rank_comparison = evaluation_results_df.pivot(
    index="molecule_id",
    columns="method",
    values="known_rank"
).reset_index()

rank_comparison["rank_change"] = (
    rank_comparison["mass_only"]
    - rank_comparison["original"]
)

rank_comparison["mass_top25"] = (
    rank_comparison["mass_only"] <= 25
)

rank_comparison["spectral_top25"] = (
    rank_comparison["original"] <= 25
)

print("CASMI 2026 - Per-Molecule Ranking Comparison")
print("---------------------------------------------")

print(
    rank_comparison[
        [
            "molecule_id",
            "mass_only",
            "original",
            "available",
            "coverage_adjusted",
            "rank_change"
        ]
    ].to_string(index=False)
)

print("\nTop-25 comparison:")

print(
    "Improved from outside to inside top 25:",
    int((
        ~rank_comparison["mass_top25"]
        & rank_comparison["spectral_top25"]
    ).sum())
)

print(
    "Dropped from inside to outside top 25:",
    int((
        rank_comparison["mass_top25"]
        & ~rank_comparison["spectral_top25"]
    ).sum())
)

print(
    "Inside top 25 with both methods:",
    int((
        rank_comparison["mass_top25"]
        & rank_comparison["spectral_top25"]
    ).sum())
)

CASMI 2026 - Per-Molecule Ranking Comparison
---------------------------------------------
molecule_id  mass_only  original  available  coverage_adjusted  rank_change
   m_145679         75         2          2                  2           73
   m_153108          2         1          1                  1            1
   m_15dab4         23        49         49                 49          -26
   m_1de28b         28         2          2                  2           26
   m_30024e         26        64         64                 64          -38
   m_7108be          8        28         28                 28          -20
   m_741f2c         22        57         57                 57          -35
   m_7b993b        125       223        223                223          -98
   m_8d19fa         19        39         39                 39          -20
   m_8d75c7          1         1          1                  1            0
   m_acdfec        254         1          1                  1          2

In [12]:
# CASMI 2026 - Diagnose candidates lost by spectral ranking

dropped = rank_comparison[
    rank_comparison["mass_top25"]
    & ~rank_comparison["spectral_top25"]
].copy()

print("CASMI 2026 - Lost Top-25 Candidates")
print("-----------------------------------")

for row in dropped.itertuples(index=False):

    molecule_id = row.molecule_id

    known_key = labels.loc[molecule_id]

    query_adducts = set(
        evaluation_queries.loc[
            evaluation_queries["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    known_refs = reference_metadata[
        reference_metadata["inchikey14"] == known_key
    ]

    matching_refs = known_refs[
        known_refs["adduct"].isin(query_adducts)
    ]

    print("\nMolecule:", molecule_id)
    print("Mass-only rank:", row.mass_only)
    print("Spectral rank:", row.original)
    print("Known structure:", known_key)
    print("Test adducts:", sorted(query_adducts))
    print("Reference spectra for known structure:", len(known_refs))
    print("References with matching adduct:", len(matching_refs))

    if not matching_refs.empty:
        print(
            "Matching reference adduct counts:",
            matching_refs["adduct"].value_counts().to_dict()
        )
    else:
        print("No matching-adduct references available.")

print("\nTotal molecules inspected:", len(dropped))

CASMI 2026 - Lost Top-25 Candidates
-----------------------------------

Molecule: m_15dab4
Mass-only rank: 23
Spectral rank: 49
Known structure: ZCHLWNAFOZCNOT
Test adducts: ['[M+H]+']
Reference spectra for known structure: 0
References with matching adduct: 0
No matching-adduct references available.

Molecule: m_7108be
Mass-only rank: 8
Spectral rank: 28
Known structure: HMBVMJFEDBMGPC
Test adducts: ['[M+H]+']
Reference spectra for known structure: 0
References with matching adduct: 0
No matching-adduct references available.

Molecule: m_741f2c
Mass-only rank: 22
Spectral rank: 57
Known structure: OXTXLZOHBPTDMJ
Test adducts: ['[M+H]+']
Reference spectra for known structure: 0
References with matching adduct: 0
No matching-adduct references available.

Molecule: m_8d19fa
Mass-only rank: 19
Spectral rank: 39
Known structure: BRJPAXQZDQTQRY
Test adducts: ['[M+Na]+']
Reference spectra for known structure: 0
References with matching adduct: 0
No matching-adduct references available.

Mol

In [13]:
# CASMI 2026 - Evaluate hybrid mass and spectral ranking

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

hybrid_results = []

ref_keys = reference_metadata["inchikey14"].to_numpy()
ref_adducts = reference_metadata["adduct"].to_numpy()

print("CASMI 2026 - Hybrid Ranking Evaluation")
print("--------------------------------------")

for molecule_id in evaluation_ids:

    queries = evaluation_queries[
        evaluation_queries["molecule_id"] == molecule_id
    ]

    query_mass = float(mass_lookup.loc[molecule_id])
    candidate_keys = evaluation_candidates[molecule_id]

    candidates = library_with_mass[
        library_with_mass["inchikey14"].isin(candidate_keys)
    ][["inchikey14", "reference_mass"]].copy()

    candidates["mass_error_ppm"] = (
        np.abs(candidates["reference_mass"] - query_mass)
        / query_mass * 1_000_000
    )

    score_sums = {key: 0.0 for key in candidate_keys}
    score_counts = {key: 0 for key in candidate_keys}

    candidate_mask = np.isin(
        ref_keys, list(candidate_keys)
    )

    # Compare spectra with the same adduct
    for adduct, query_group in queries.groupby("adduct"):

        positions = np.flatnonzero(
            candidate_mask & (ref_adducts == adduct)
        )

        if len(positions) == 0:
            continue

        X_query = np.stack([
            make_feature_vector(row)
            for row in query_group.itertuples(index=False)
        ])

        X_ref = reference_features[positions]

        similarities = cosine_similarity(X_query, X_ref)
        selected_keys = ref_keys[positions]

        for key in np.unique(selected_keys):

            key_positions = np.flatnonzero(
                selected_keys == key
            )

            best_scores = similarities[
                :, key_positions
            ].max(axis=1)

            score_sums[key] += float(best_scores.sum())
            score_counts[key] += len(best_scores)

    # Original spectral scores
    candidates["spectral_score"] = candidates[
        "inchikey14"
    ].map(
        lambda key: (
            score_sums[key] / len(queries)
            if score_counts[key] > 0
            else np.nan
        )
    )

    # Rank every candidate by mass
    mass_order = candidates.sort_values(
        ["mass_error_ppm", "inchikey14"]
    ).reset_index(drop=True)

    mass_ranks = {
        key: rank
        for rank, key in enumerate(
            mass_order["inchikey14"], start=1
        )
    }

    # Rank only candidates with spectral evidence
    spectral_order = candidates.dropna(
        subset=["spectral_score"]
    ).sort_values(
        ["spectral_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True]
    )

    spectral_ranks = {
        key: rank
        for rank, key in enumerate(
            spectral_order["inchikey14"], start=1
        )
    }

    # No spectral reference: retain the mass rank
    # instead of treating missing evidence as poor similarity.
    def hybrid_score(key):
        mass_rank = mass_ranks[key]
        spectral_rank = spectral_ranks.get(key, mass_rank)

        return (
            0.7 / (10 + mass_rank)
            + 0.3 / (10 + spectral_rank)
        )

    candidates["hybrid_score"] = candidates[
        "inchikey14"
    ].map(hybrid_score)

    hybrid_order = candidates.sort_values(
        ["hybrid_score", "mass_error_ppm", "inchikey14"],
        ascending=[False, True, True]
    ).reset_index(drop=True)

    # Use the known structure ONLY after ranking
    known_key = labels.loc[molecule_id]

    positions = np.flatnonzero(
        hybrid_order["inchikey14"].to_numpy() == known_key
    )

    known_rank = (
        int(positions[0]) + 1
        if len(positions) == 1
        else None
    )

    hybrid_results.append({
        "molecule_id": molecule_id,
        "known_rank": known_rank,
        "top1": known_rank == 1,
        "top25": known_rank is not None and known_rank <= 25,
        "reciprocal_rank_at_25": (
            1.0 / known_rank
            if known_rank is not None and known_rank <= 25
            else 0.0
        )
    })

hybrid_results_df = pd.DataFrame(hybrid_results)

print("\nCASMI 2026 - Hybrid Results")
print("---------------------------")
print("Molecules evaluated:", len(hybrid_results_df))
print("Top 1:", int(hybrid_results_df["top1"].sum()))
print("Top 25:", int(hybrid_results_df["top25"].sum()))
print(
    "MRR@25:",
    round(
        hybrid_results_df["reciprocal_rank_at_25"].mean(),
        6
    )
)

print("\nRanks of the 7 previously lost molecules:")

print(
    hybrid_results_df[
        hybrid_results_df["molecule_id"].isin(
            dropped["molecule_id"]
        )
    ][["molecule_id", "known_rank"]].to_string(index=False)
)

CASMI 2026 - Hybrid Ranking Evaluation
--------------------------------------

CASMI 2026 - Hybrid Results
---------------------------
Molecules evaluated: 20
Top 1: 3
Top 25: 16
MRR@25: 0.233314

Ranks of the 7 previously lost molecules:
molecule_id  known_rank
   m_f866a0           5
   m_e138be          22
   m_8d19fa          20
   m_7108be          10
   m_dc7be7          12
   m_15dab4          30
   m_741f2c          24


In [14]:
# CASMI 2026 - Compare hybrid and mass-only rankings

import pandas as pd

# Get mass-only ranks from our previous evaluation
mass_ranks = evaluation_results_df[
    evaluation_results_df["method"] == "mass_only"
][["molecule_id", "known_rank"]].rename(
    columns={"known_rank": "mass_rank"}
)

# Get hybrid ranks
hybrid_ranks = hybrid_results_df[
    ["molecule_id", "known_rank"]
].rename(
    columns={"known_rank": "hybrid_rank"}
)

comparison = mass_ranks.merge(
    hybrid_ranks,
    on="molecule_id",
    validate="one_to_one"
)

comparison["mass_top25"] = comparison["mass_rank"] <= 25
comparison["hybrid_top25"] = comparison["hybrid_rank"] <= 25

print("CASMI 2026 - Hybrid vs Mass-Only")
print("--------------------------------")

print(
    "Improved into Top 25:",
    int((
        ~comparison["mass_top25"]
        & comparison["hybrid_top25"]
    ).sum())
)

print(
    "Dropped out of Top 25:",
    int((
        comparison["mass_top25"]
        & ~comparison["hybrid_top25"]
    ).sum())
)

print(
    "Inside Top 25 with both:",
    int((
        comparison["mass_top25"]
        & comparison["hybrid_top25"]
    ).sum())
)

print("\nMolecules improved into Top 25:")
print(
    comparison.loc[
        ~comparison["mass_top25"]
        & comparison["hybrid_top25"],
        ["molecule_id", "mass_rank", "hybrid_rank"]
    ].to_string(index=False)
)

print("\nMolecules dropped out of Top 25:")
print(
    comparison.loc[
        comparison["mass_top25"]
        & ~comparison["hybrid_top25"],
        ["molecule_id", "mass_rank", "hybrid_rank"]
    ].to_string(index=False)
)

CASMI 2026 - Hybrid vs Mass-Only
--------------------------------
Improved into Top 25: 4
Dropped out of Top 25: 1
Inside Top 25 with both: 12

Molecules improved into Top 25:
molecule_id  mass_rank  hybrid_rank
   m_bf6cfe         54           15
   m_1de28b         28           10
   m_acdfec        254           18
   m_145679         75           16

Molecules dropped out of Top 25:
molecule_id  mass_rank  hybrid_rank
   m_15dab4         23           30


In [15]:
# CASMI 2026 - Prepare an independent evaluation subset

import numpy as np

rng = np.random.default_rng(123)

# Exclude all 20 molecules used in our previous experiments
available_ids = sorted(
    set(sample_submission["molecule_id"])
    - set(evaluation_ids)
)

holdout_ids = rng.choice(
    available_ids,
    size=30,
    replace=False
).tolist()

holdout_queries = test_spectra[
    test_spectra["molecule_id"].isin(holdout_ids)
].copy()

# Verify that the two evaluation groups do not overlap
overlap = set(holdout_ids) & set(evaluation_ids)

print("CASMI 2026 - Independent Evaluation Subset")
print("------------------------------------------")
print("Previous evaluation molecules:", len(evaluation_ids))
print("New evaluation molecules:", len(holdout_ids))
print("New evaluation test spectra:", len(holdout_queries))
print("Overlapping molecules:", len(overlap))

assert len(overlap) == 0
assert len(holdout_ids) == 30

print("\nNew evaluation subset prepared successfully!")

CASMI 2026 - Independent Evaluation Subset
------------------------------------------
Previous evaluation molecules: 20
New evaluation molecules: 30
New evaluation test spectra: 90
Overlapping molecules: 0

New evaluation subset prepared successfully!


In [16]:
# CASMI 2026 - Prepare candidates for the 30-molecule holdout

import numpy as np

PPM_TOLERANCE = 20

holdout_candidates = {}
holdout_candidate_keys = set()
holdout_adducts = set()

# Reuse the candidate library and masses already in memory
reference_masses = library_with_mass[
    "reference_mass"
].to_numpy(dtype=float)

for molecule_id in holdout_ids:

    query_mass = float(mass_lookup.loc[molecule_id])

    ppm_errors = (
        np.abs(reference_masses - query_mass)
        / query_mass
    ) * 1_000_000

    candidate_keys = set(
        library_with_mass.loc[
            ppm_errors <= PPM_TOLERANCE,
            "inchikey14"
        ]
    )

    holdout_candidates[molecule_id] = candidate_keys
    holdout_candidate_keys.update(candidate_keys)

    molecule_adducts = set(
        holdout_queries.loc[
            holdout_queries["molecule_id"] == molecule_id,
            "adduct"
        ]
    )

    holdout_adducts.update(molecule_adducts)

print("CASMI 2026 - Holdout Candidates")
print("--------------------------------")
print("Holdout molecules:", len(holdout_candidates))
print("Holdout test spectra:", len(holdout_queries))
print(
    "Unique candidate structures:",
    len(holdout_candidate_keys)
)
print(
    "Minimum candidates per molecule:",
    min(len(keys) for keys in holdout_candidates.values())
)
print(
    "Maximum candidates per molecule:",
    max(len(keys) for keys in holdout_candidates.values())
)
print("Adducts:", sorted(holdout_adducts))

assert len(holdout_candidates) == 30

print("\nHoldout candidates prepared successfully!")

CASMI 2026 - Holdout Candidates
--------------------------------
Holdout molecules: 30
Holdout test spectra: 90
Unique candidate structures: 5172
Minimum candidates per molecule: 30
Maximum candidates per molecule: 381
Adducts: ['[M+H]+', '[M+NH4]+', '[M+Na]+', '[M-H]-']

Holdout candidates prepared successfully!


In [17]:
# CASMI 2026 - Collect compact references for the 30-molecule holdout

import gc
import os
import numpy as np
import pandas as pd
import psutil

process = psutil.Process(os.getpid())

BATCH_SIZE = 5000
MAX_PROCESS_RAM_GB = 6.0
MIN_AVAILABLE_RAM_GB = 6.0
MAX_REFERENCES = 60000

reference_columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Exclude exact copies of the 90 holdout test spectra
holdout_fingerprints = {
    spectrum_fingerprint(row)
    for row in holdout_queries.itertuples(index=False)
}

holdout_feature_parts = []
holdout_ref_keys = []
holdout_ref_adducts = []

excluded_copies = 0
rows_inspected = 0
collection_complete = True
stop_reason = None

print("CASMI 2026 - Holdout Reference Collection")
print("-----------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=BATCH_SIZE,
        columns=reference_columns
    ):
        batch_df = batch.to_pandas()
        rows_inspected += len(batch_df)

        matches = batch_df[
            batch_df["inchikey14"].isin(holdout_candidate_keys)
            & batch_df["adduct"].isin(holdout_adducts)
        ]

        for row in matches.itertuples(index=False):

            if spectrum_fingerprint(row) in holdout_fingerprints:
                excluded_copies += 1
                continue

            if len(holdout_feature_parts) >= MAX_REFERENCES:
                collection_complete = False
                stop_reason = "Reference count limit reached"
                break

            holdout_feature_parts.append(
                make_feature_vector(row)
            )
            holdout_ref_keys.append(row.inchikey14)
            holdout_ref_adducts.append(row.adduct)

        del matches, batch_df, batch

        used_ram = process.memory_info().rss / (1024 ** 3)
        available_ram = (
            psutil.virtual_memory().available / (1024 ** 3)
        )

        if (
            used_ram >= MAX_PROCESS_RAM_GB
            or available_ram <= MIN_AVAILABLE_RAM_GB
        ):
            collection_complete = False
            stop_reason = "RAM safety limit reached"

        if not collection_complete:
            break

    print(
        f"Row groups checked: {group_index + 1}/"
        f"{train_file.num_row_groups} | "
        f"References: {len(holdout_feature_parts)} | "
        f"RAM: {used_ram:.2f} GB"
    )

    gc.collect()

    if not collection_complete:
        break


print("\nCASMI 2026 - Holdout Collection Results")
print("---------------------------------------")
print("Collection complete:", collection_complete)
print("Rows inspected:", rows_inspected)
print("References collected:", len(holdout_feature_parts))
print("Exact copies excluded:", excluded_copies)
print(
    "Current RAM:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)

if collection_complete and holdout_feature_parts:

    holdout_reference_features = np.stack(
        holdout_feature_parts
    )

    holdout_reference_metadata = pd.DataFrame({
        "inchikey14": holdout_ref_keys,
        "adduct": holdout_ref_adducts
    })

    holdout_output_path = (
        "/kaggle/working/casmi_holdout_references.npz"
    )

    np.savez_compressed(
        holdout_output_path,
        features=holdout_reference_features,
        inchikey14=np.asarray(holdout_ref_keys),
        adduct=np.asarray(holdout_ref_adducts),
        holdout_ids=np.asarray(holdout_ids)
    )

    print(
        "Feature matrix shape:",
        holdout_reference_features.shape
    )
    print(
        "Feature matrix size:",
        round(
            holdout_reference_features.nbytes / (1024 ** 2),
            2
        ),
        "MB"
    )
    print("Saved file:", holdout_output_path)

else:
    print("Stopped early:", stop_reason)
    print("Incomplete reference data was not saved.")

del holdout_feature_parts
gc.collect()

CASMI 2026 - Holdout Reference Collection
-----------------------------------------
Row groups checked: 1/21 | References: 2197 | RAM: 1.10 GB
Row groups checked: 2/21 | References: 4351 | RAM: 1.10 GB
Row groups checked: 3/21 | References: 6607 | RAM: 1.10 GB
Row groups checked: 4/21 | References: 8835 | RAM: 1.11 GB
Row groups checked: 5/21 | References: 10969 | RAM: 1.11 GB
Row groups checked: 6/21 | References: 13039 | RAM: 1.11 GB
Row groups checked: 7/21 | References: 15012 | RAM: 1.12 GB
Row groups checked: 8/21 | References: 17361 | RAM: 1.13 GB
Row groups checked: 9/21 | References: 19058 | RAM: 1.14 GB
Row groups checked: 10/21 | References: 19560 | RAM: 1.14 GB
Row groups checked: 11/21 | References: 20209 | RAM: 1.14 GB
Row groups checked: 12/21 | References: 20648 | RAM: 1.15 GB
Row groups checked: 13/21 | References: 21659 | RAM: 1.15 GB
Row groups checked: 14/21 | References: 23025 | RAM: 1.15 GB
Row groups checked: 15/21 | References: 24345 | RAM: 1.16 GB
Row groups che

0

In [18]:
# CASMI 2026 - Collect compact references for the 30-molecule holdout

import gc
import os
import numpy as np
import pandas as pd
import psutil

process = psutil.Process(os.getpid())

BATCH_SIZE = 5000
MAX_PROCESS_RAM_GB = 6.0
MIN_AVAILABLE_RAM_GB = 6.0
MAX_REFERENCES = 60000

reference_columns = [
    "inchikey14",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

# Exclude exact copies of the 90 holdout test spectra
holdout_fingerprints = {
    spectrum_fingerprint(row)
    for row in holdout_queries.itertuples(index=False)
}

holdout_feature_parts = []
holdout_ref_keys = []
holdout_ref_adducts = []

excluded_copies = 0
rows_inspected = 0
collection_complete = True
stop_reason = None

print("CASMI 2026 - Holdout Reference Collection")
print("-----------------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=BATCH_SIZE,
        columns=reference_columns
    ):
        batch_df = batch.to_pandas()
        rows_inspected += len(batch_df)

        matches = batch_df[
            batch_df["inchikey14"].isin(holdout_candidate_keys)
            & batch_df["adduct"].isin(holdout_adducts)
        ]

        for row in matches.itertuples(index=False):

            if spectrum_fingerprint(row) in holdout_fingerprints:
                excluded_copies += 1
                continue

            if len(holdout_feature_parts) >= MAX_REFERENCES:
                collection_complete = False
                stop_reason = "Reference count limit reached"
                break

            holdout_feature_parts.append(
                make_feature_vector(row)
            )
            holdout_ref_keys.append(row.inchikey14)
            holdout_ref_adducts.append(row.adduct)

        del matches, batch_df, batch

        used_ram = process.memory_info().rss / (1024 ** 3)
        available_ram = (
            psutil.virtual_memory().available / (1024 ** 3)
        )

        if (
            used_ram >= MAX_PROCESS_RAM_GB
            or available_ram <= MIN_AVAILABLE_RAM_GB
        ):
            collection_complete = False
            stop_reason = "RAM safety limit reached"

        if not collection_complete:
            break

    print(
        f"Row groups checked: {group_index + 1}/"
        f"{train_file.num_row_groups} | "
        f"References: {len(holdout_feature_parts)} | "
        f"RAM: {used_ram:.2f} GB"
    )

    gc.collect()

    if not collection_complete:
        break


print("\nCASMI 2026 - Holdout Collection Results")
print("---------------------------------------")
print("Collection complete:", collection_complete)
print("Rows inspected:", rows_inspected)
print("References collected:", len(holdout_feature_parts))
print("Exact copies excluded:", excluded_copies)
print(
    "Current RAM:",
    round(process.memory_info().rss / (1024 ** 3), 2),
    "GB"
)

if collection_complete and holdout_feature_parts:

    holdout_reference_features = np.stack(
        holdout_feature_parts
    )

    holdout_reference_metadata = pd.DataFrame({
        "inchikey14": holdout_ref_keys,
        "adduct": holdout_ref_adducts
    })

    holdout_output_path = (
        "/kaggle/working/casmi_holdout_references.npz"
    )

    np.savez_compressed(
        holdout_output_path,
        features=holdout_reference_features,
        inchikey14=np.asarray(holdout_ref_keys),
        adduct=np.asarray(holdout_ref_adducts),
        holdout_ids=np.asarray(holdout_ids)
    )

    print(
        "Feature matrix shape:",
        holdout_reference_features.shape
    )
    print(
        "Feature matrix size:",
        round(
            holdout_reference_features.nbytes / (1024 ** 2),
            2
        ),
        "MB"
    )
    print("Saved file:", holdout_output_path)

else:
    print("Stopped early:", stop_reason)
    print("Incomplete reference data was not saved.")

del holdout_feature_parts
gc.collect()

CASMI 2026 - Holdout Reference Collection
-----------------------------------------
Row groups checked: 1/21 | References: 2197 | RAM: 1.30 GB
Row groups checked: 2/21 | References: 4351 | RAM: 1.30 GB
Row groups checked: 3/21 | References: 6607 | RAM: 1.30 GB
Row groups checked: 4/21 | References: 8835 | RAM: 1.30 GB
Row groups checked: 5/21 | References: 10969 | RAM: 1.30 GB
Row groups checked: 6/21 | References: 13039 | RAM: 1.30 GB
Row groups checked: 7/21 | References: 15012 | RAM: 1.30 GB
Row groups checked: 8/21 | References: 17361 | RAM: 1.30 GB
Row groups checked: 9/21 | References: 19058 | RAM: 1.30 GB
Row groups checked: 10/21 | References: 19560 | RAM: 1.30 GB
Row groups checked: 11/21 | References: 20209 | RAM: 1.30 GB
Row groups checked: 12/21 | References: 20648 | RAM: 1.30 GB
Row groups checked: 13/21 | References: 21659 | RAM: 1.30 GB
Row groups checked: 14/21 | References: 23025 | RAM: 1.30 GB
Row groups checked: 15/21 | References: 24345 | RAM: 1.30 GB
Row groups che

0

In [19]:
# CASMI 2026 - Restore labels for the 30-molecule holdout

from collections import defaultdict
import pandas as pd

# Link each holdout spectrum fingerprint to its molecule ID
holdout_lookup = defaultdict(set)

for row in holdout_queries.itertuples(index=False):
    holdout_lookup[spectrum_fingerprint(row)].add(
        row.molecule_id
    )

# Avoid fingerprinting training spectra with irrelevant adducts
# and peak counts.
holdout_combinations = {
    (row.adduct, len(row.ms2_mzs))
    for row in holdout_queries.itertuples(index=False)
}

holdout_label_hits = defaultdict(set)

columns = [
    "inchikey14",
    "normalized_smiles",
    "adduct",
    "ms2_mzs",
    "ms2_normalized_intensities"
]

print("CASMI 2026 - Holdout Label Recovery")
print("-----------------------------------")

for group_index in range(train_file.num_row_groups):

    for batch in train_file.iter_batches(
        row_groups=[group_index],
        batch_size=5000,
        columns=columns
    ):
        batch_df = batch.to_pandas()

        for row in batch_df.itertuples(index=False):

            if (
                row.adduct,
                len(row.ms2_mzs)
            ) not in holdout_combinations:
                continue

            fingerprint = spectrum_fingerprint(row)

            for molecule_id in holdout_lookup.get(
                fingerprint, ()
            ):
                holdout_label_hits[molecule_id].add(
                    (row.inchikey14, row.normalized_smiles)
                )

    if (
        (group_index + 1) % 5 == 0
        or group_index + 1 == train_file.num_row_groups
    ):
        print(
            f"Row groups checked: {group_index + 1}/"
            f"{train_file.num_row_groups}"
        )


# Check whether each molecule has one unambiguous structure
unique_labels = {
    molecule_id: next(iter(structures))
    for molecule_id, structures in holdout_label_hits.items()
    if len(structures) == 1
}

ambiguous_count = sum(
    len(structures) > 1
    for structures in holdout_label_hits.values()
)

print("\nCASMI 2026 - Holdout Label Results")
print("----------------------------------")
print("Holdout molecules:", len(holdout_ids))
print("Molecules with one structure:", len(unique_labels))
print("Molecules with multiple structures:", ambiguous_count)
print(
    "Molecules without a match:",
    len(holdout_ids) - len(holdout_label_hits)
)

if len(unique_labels) == len(holdout_ids):

    holdout_label_df = pd.DataFrame([
        {
            "molecule_id": molecule_id,
            "inchikey14": unique_labels[molecule_id][0],
            "normalized_smiles": unique_labels[molecule_id][1]
        }
        for molecule_id in holdout_ids
    ])

    labels_path = (
        "/kaggle/working/casmi_holdout_labels.csv"
    )

    holdout_label_df.to_csv(labels_path, index=False)

    print("\nLabels saved:", labels_path)

else:
    print(
        "\nSome labels are missing or ambiguous. "
        "Review them before evaluating the model."
    )

CASMI 2026 - Holdout Label Recovery
-----------------------------------
Row groups checked: 5/21
Row groups checked: 10/21
Row groups checked: 15/21
Row groups checked: 20/21
Row groups checked: 21/21

CASMI 2026 - Holdout Label Results
----------------------------------
Holdout molecules: 30
Molecules with one structure: 30
Molecules with multiple structures: 0
Molecules without a match: 0

Labels saved: /kaggle/working/casmi_holdout_labels.csv
